# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the published FAIR\^2 Croissant-structured dataset using the [mlcroissant](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset is published with a Croissant schema at:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and inspect its basic properties using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata

print("Dataset title:", metadata.name)
print("Description:\n", metadata.description)

## 2. Data Overview
Enumerate available record sets (`@id`), fields within each record set, and their associated columns. This is done by accessing metadata via `mlcroissant.dataset.metadata.record_sets`.

We will print their unique `@id` values for all exploration and extraction steps.

In [ ]:
# List all record sets and their fields by @id

record_sets = dataset.metadata.record_sets
print(f"Total {len(record_sets)} record set(s) found:\n")
for rs in record_sets:
    print(f"Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else '-'}")
    print(f"  Fields:")
    for f in rs.fields:
        print(f"    - {f.name} (field @id: {f.id}, column @id: {f.column.id})")
    print()

## 3. Data Extraction
Select a record set to extract data. We'll load the main protein quantification table, which contains normalized protein abundances and metadata, using the relevant record set `@id`.
Update the variable `main_record_set_id` with the appropriate value found above.

In [ ]:
# Update this @id to the primary record set (table) you want to explore
# Example: main_record_set_id = 'https://api.app.sen.science/frontiers/7154140/7c9e4a7c-fb21-4d6b-833b-cf61f7eea6b5'
main_record_set_id = dataset.metadata.record_sets[0].id  # choose the first one for example

def records_preview(ds, record_set_id, n=5):
    print(f"Sample records from {record_set_id}:")
    for i, row in enumerate(ds.records(record_set=record_set_id)):
        print(row)
        if i+1 >= n:
            break

records_preview(dataset, main_record_set_id, n=3)

Now, let's load **all records** of this table into a DataFrame and display the columns. We'll use the record set's `@id`.

In [ ]:
df = pd.DataFrame(dataset.records(record_set=main_record_set_id))
print(f"Shape: {df.shape}")
print("Columns:", list(df.columns))
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field by its `@id` for further filtering and transformation. We'll demonstrate with one numeric field. To find available fields and their `@id`, refer to the Data Overview section above.

In [ ]:
# Replace with the actual field id for a numeric value, e.g. 'cr:normalized_abundance'
# For illustration, pick the first numeric field from the selected record set
selected_record_set = None
for rs in dataset.metadata.record_sets:
    if rs.id == main_record_set_id:
        selected_record_set = rs
        break
numeric_field = None
for f in selected_record_set.fields:
    # Heuristically select the first numeric field
    if hasattr(f, 'data_type') and f.data_type in ['Float', 'Number', 'Integer']:
        numeric_field = f.id  # Use field @id
        numeric_field_name = f.name
        break
if numeric_field is None:
    raise Exception("No numeric field found in the chosen record set.")

print(f"Numeric field selected: {numeric_field_name} (field @id: {numeric_field})")
# Filter records where this numeric field > 10 (if suitable; adjust threshold for context)
try:
    df_num = df.astype({numeric_field: float})
except:
    df_num = df.copy()
    # Try pandas to_numeric
    df_num[numeric_field] = pd.to_numeric(df_num[numeric_field], errors='coerce')

threshold = 10
filtered_df = df_num[df_num[numeric_field] > threshold]
print(f"Filtered records with {numeric_field_name} (@id:{numeric_field}) > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
from numpy import nanmean, nanstd
mu = nanmean(filtered_df[numeric_field])
sigma = nanstd(filtered_df[numeric_field])
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mu) / sigma
print(f"\nNormalized {numeric_field_name} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Optionally group by a categorical field (e.g. 'cr:sample_id')
group_field = None
for f in selected_record_set.fields:
    # Find a field that is likely to be categorical but not the numeric one
    if f.id != numeric_field and (getattr(f, 'data_type', None) == 'Text' or getattr(f, 'data_type', None) == 'String' or 'id' in f.name.lower()):
        group_field = f.id
        group_field_name = f.name
        break
if group_field and group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nMean {numeric_field_name} grouped by {group_field_name} (field @id: {group_field}):")
    print(grouped_df.head())

## 5. Visualization
Plot the distribution of the selected numeric field and a groupwise boxplot if a group field was selected.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of numeric field
plt.figure(figsize=(7,4))
filtered_df[numeric_field].hist(bins=30)
plt.title(f"Distribution of {numeric_field_name}")
plt.xlabel(numeric_field_name)
plt.ylabel("Count")
plt.show()

# Boxplot by group field
if group_field and group_field in filtered_df.columns:
    plt.figure(figsize=(12,5))
    filtered_df.boxplot(column=numeric_field, by=group_field)
    plt.title(f"{numeric_field_name} by {group_field_name}")
    plt.suptitle('')
    plt.ylabel(numeric_field_name)
    plt.xlabel(group_field_name)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we:
- Loaded mass spectrometry-based protein abundance data using the Croissant open metadata schema and the `mlcroissant` library.
- Explored available tables (record sets), fields, and columns by their unique `@id` values.
- Selected and analyzed numerical fields, filtered and normalized abundance measurements, and explored distributions across experimental groups.

**Next steps:** Further biological or statistical analysis, comparison of stimulation effects on protein abundances, or integrating external protein annotation sources.